In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
from scipy.ndimage import label
from pathlib import Path


# utils functions for loading data, plotting grids, and finding objects in grids
def load_arc_data(data_path):
    with open(data_path) as f:
        return json.load(f)

def plot_grid(grid, title="Grid"):
    cmap = colors.ListedColormap(['#000', '#0074D9', '#FF4136', '#2ECC40', '#FFDC00',
                                  '#AAAAAA', '#F012BE', '#FF851B', '#7FDBFF', '#870C25'])
    norm = colors.BoundaryNorm(np.arange(-0.5, 10, 1), cmap.N)
    plt.imshow(grid, cmap=cmap, norm=norm)
    plt.title(title)
    plt.axis('off')


def find_objects(grid):
    grid = np.array(grid)
    binary_grid = (grid > 0).astype(int)
    
    labeled_array, num_features = label(binary_grid)
    
    objects = []
    for i in range(1, num_features + 1):
        coords = np.argwhere(labeled_array == i)
        objects.append(coords)
        
    return objects



# solver functions
def repeat_pattern(input_grid, n_times):
    """Tiles the input grid n_times in both dimensions."""
    arr = np.array(input_grid)
    output = np.tile(arr, (n_times, n_times))
    return output.tolist()

def translate_object(grid, obj_coords, dr, dc):
    """
    Moves an object by dr (rows) and dc (columns).
    dr: positive = down, negative = up
    dc: positive = right, negative = left
    """
    grid = np.array(grid)
    new_grid = np.zeros_like(grid)
    
    for r, c in obj_coords:
        new_r, new_c = r + dr, c + dc
        if 0 <= new_r < grid.shape[0] and 0 <= new_c < grid.shape[1]:
            new_grid[new_r, new_c] = grid[r, c]
            
    return new_grid.tolist()

def rotate_grid(grid, k=1):
    """Rotates grid 90 degrees k times."""
    return np.rot90(np.array(grid), k=k).tolist()

def flip_grid(grid, axis=0):
    """Flips grid: axis=0 is vertical (up-down), axis=1 is horizontal (left-right)."""
    if axis == 0:
        return np.flipud(np.array(grid)).tolist()
    else:
        return np.fliplr(np.array(grid)).tolist()

def kronecker_solve(grid):
    """Applies Kronecker product - essential for task 007bbfb7."""
    arr = np.array(grid)
    res = np.kron(arr > 0, arr)
    return res.tolist()

def master_solver(train_tasks, test_input):
    """
    The Brain: Tests various logical transformations against training data
    to find a consistent rule, then applies it to the test data.
    """
    
    # Define candidate logics with descriptive labels to avoid lambda __name__ errors
    candidate_logics = [
        ("Rotate 90", lambda x: rotate_grid(x, k=1)),
        ("Rotate 180", lambda x: rotate_grid(x, k=2)),
        ("Flip Vertical", lambda x: flip_grid(x, axis=0)),
        ("Flip Horizontal", lambda x: flip_grid(x, axis=1)),
        ("Repeat 3x", lambda x: repeat_pattern(x, 3)),
        ("Kronecker Product", lambda x: kronecker_solve(x))
    ]
    
    for label_name, logic_func in candidate_logics:
        is_correct = True
        
        for example in train_tasks:
            inp, out = example['input'], example['output']
            try:
                if logic_func(inp) != out:
                    is_correct = False
                    break
            except:
                is_correct = False
                break
        
        if is_correct:
            print(f"Success! Found winning logic: {label_name}")
            return logic_func(test_input)
            
    return "No winning logic found."



# submission = master_solver(train_tasks, test_input)
def generate_submission(data_path, output_path):
    """
    Automates the solving process for all tasks in a dataset.
    1. Loads the challenge data.
    2. Iterates through every task.
    3. Uses master_solver to find a solution.
    4. Formats and saves the final JSON.
    """
    # Load the challenges
    data = load_arc_data(data_path)
    submission = {}

    print(f"Starting submission generation for {len(data)} tasks...")

    for task_id, task_content in data.items():
        train_examples = task_content['train']
        # Each test task can have multiple test inputs, though usually it's one
        test_inputs = task_content['test']
        
        task_predictions = []
        for test_item in test_inputs:
            prediction = master_solver(train_examples, test_item['input'])
            
            # If solver fails, provide a dummy black 1x1 grid as a placeholder
            if isinstance(prediction, str):
                prediction = [[0]]
                
            task_predictions.append({"attempt_1": prediction})
        
        submission[task_id] = task_predictions

    # Write final results to submission.json
    with open(output_path, 'w') as f:
        json.dump(submission, f)
    
    print(f"Submission saved to {output_path}")

if __name__ == "__main__":
    # Path to the specific challenge file provided by Kaggle
    test_file = Path("../data/arc-agi_test_challenges.json")
    generate_submission(test_file, "submission.json")